In [13]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

In [14]:
LOG_DIR = "../logs/"

AGG_LOG = "he-aggregation-service"
CLIENT1_LOG = "he-client-1-go"
CLIENT2_LOG = "he-client-2-go"

In [15]:
agg_time = pd.read_csv(f"{LOG_DIR}{AGG_LOG}-data.csv")
client1_time = pd.read_csv(f"{LOG_DIR}{CLIENT1_LOG}-data.csv")
client2_time = pd.read_csv(f"{LOG_DIR}{CLIENT2_LOG}-data.csv")

agg_time["Service"] = "AggregationService"
client1_time["Service"] = "Client1"
client2_time["Service"] = "Client2"

client1_time.head()

data_joined = pd.concat(
    [agg_time, client1_time, client2_time],
    axis=0)

data_joined.sort_values(
    by=["EndTime", "Service"],
    inplace=True)
data_joined.reset_index(drop=True, inplace=True)

def create_timeline_offset(df):
    """Create a timeline with an offset for the x-axis."""
    min_time = df["EndTime"].min()
    df["StartMs"] = (pd.to_datetime(df["EndTime"], format="%H:%M:%S.%f") - pd.to_datetime(min_time, format="%H:%M:%S.%f")).dt.total_seconds() * 1000
    
    return df

data_joined = create_timeline_offset(data_joined)

data_joined

,Date,EndTime,Name,Alloc,TotalAlloc,Sys,Service,StartMs
0,2025/07/22,16:42:45.841763,ClientManagmentHandler - handleRegisterClient,25,40,33,AggregationService,0.000
1,2025/07/22,16:42:45.844413,ClientManagmentHandler - handleRegisterClient,23,45,41,AggregationService,2.650
2,2025/07/22,16:42:45.854954,HEService - SetParameters,9,13,21,Client1,13.191
3,2025/07/22,16:42:45.863301,HEService - SetParameters - after public key g...,44,69,65,Client1,21.538
4,2025/07/22,16:42:50.003130,UpdateClients - Initiated,23,45,41,AggregationService,4161.367
...,...,...,...,...,...,...,...,...
153,2025/07/22,16:44:17.591467,ModelHandler - handleUpdateModel - after response,973,8538,1724,Client1,91749.704
154,2025/07/22,16:44:17.681800,ModelHandler - handleUpdateModel,1056,7688,1792,Client2,91840.037
155,2025/07/22,16:44:18.698766,ModelHandler - handleUpdateModel - after response,1128,7760,1792,Client2,92857.003
156,2025/07/22,16:44:18.698964,UpdateClients - AfterUpdateClients,834,14674,2125,AggregationService,92857.201


In [16]:
data_joined["MethodPart"] = 0
data_joined.loc[data_joined["Name"].str.contains("after"), "MethodPart"] = 1
data_joined.head()

,Date,EndTime,Name,Alloc,TotalAlloc,Sys,Service,StartMs,MethodPart
0,2025/07/22,16:42:45.841763,ClientManagmentHandler - handleRegisterClient,25,40,33,AggregationService,0.000,0
1,2025/07/22,16:42:45.844413,ClientManagmentHandler - handleRegisterClient,23,45,41,AggregationService,2.650,0
2,2025/07/22,16:42:45.854954,HEService - SetParameters,9,13,21,Client1,13.191,0
3,2025/07/22,16:42:45.863301,HEService - SetParameters - after public key g...,44,69,65,Client1,21.538,1
4,2025/07/22,16:42:50.003130,UpdateClients - Initiated,23,45,41,AggregationService,4161.367,0


In [17]:
print(pio.templates)

pio.templates["research"] = go.layout.Template(
    layout=dict(
        font=dict(color="#000000"),
        paper_bgcolor="#ffffff",
        plot_bgcolor="#ffffff",
        hovermode="closest",
        xaxis=dict(
            tickangle=-45,
            showline=True,
            linewidth=1,
            linecolor="#373737",
            ticks="inside",
            showgrid=True,
            gridcolor="#373737",
            zeroline=True,
            zerolinecolor="#373737",
            zerolinewidth=1,
            mirror=True
        ),
        yaxis=dict(
            showline=True,
            linewidth=1,
            linecolor="#373737",
            ticks="inside",
            showgrid=True,
            gridcolor="#373737",
            zeroline=True,
            zerolinecolor="#373737",
            zerolinewidth=1,
            mirror=True
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.5,
            xanchor="right",
            x=0.95
        ),
    )
)

pio.templates.default = "plotly_white+research"

Templates configuration
-----------------------
    Default template: 'plotly_white+research'
    Available templates:
        ['ggplot2', 'seaborn', 'simple_white', 'plotly',
         'plotly_white', 'plotly_dark', 'presentation', 'xgridoff',
         'ygridoff', 'gridon', 'none', 'research']



In [21]:
import plotly.express as px
import plotly.graph_objects as go


df = px.data.tips()
fig = px.line(data_joined, x="StartMs", y="Alloc", color='Service', 
             hover_data=["EndTime", "Alloc", "TotalAlloc", "Service"], labels={"EndTime": "Time", "Service": "Service", "Alloc": "Alloc (MB)"}, color_discrete_sequence=px.colors.qualitative.Safe,
             height=400)

def add_rect_annotation(rect, text, color="lightblue", opacity=0.6):
    """Add a rectangle and annotation to the figure."""
    fig.add_shape(
        type="rect", x0=rect[0], y0=rect[1], x1=rect[2], y1=rect[3],
        fillcolor=color, opacity=opacity,
        line=dict(color=color, width=1),
    )
    fig.add_annotation(
        x=rect[0] + (rect[2]-rect[0])/2,  # Center of the rectangle
        y=rect[1] + (rect[3]-rect[1])/2,  # Position above the rectangle
        text=text,
        showarrow=False,
        font=dict(size=12, color="black"),
        opacity=0.8
    )

y_coor = (1300, 1500)

add_rect_annotation((3000, y_coor[0], 15000, y_coor[1]), "Client Registering", color="lightblue")
add_rect_annotation((15000, y_coor[0], 30000, y_coor[1]), "Training", color="lightgreen")
add_rect_annotation((30000, y_coor[0], 38000, y_coor[1]), "Enc", color="lightcoral")
add_rect_annotation((38000, y_coor[0], 42800, y_coor[1]), "Agg", color="lightyellow")
add_rect_annotation((42800, y_coor[0], 57000, y_coor[1]), "Training", color="lightgreen")
add_rect_annotation((57000, y_coor[0], 64000, y_coor[1]), "Enc", color="lightcoral")
add_rect_annotation((64000, y_coor[0], 69000, y_coor[1]), "Agg", color="lightyellow")
add_rect_annotation((69000, y_coor[0], 84000, y_coor[1]), "Training", color="lightgreen")
add_rect_annotation((84000, y_coor[0], 89500, y_coor[1]), "Enc", color="lightcoral")
add_rect_annotation((89500, y_coor[0], 95000, y_coor[1]), "Agg", color="lightyellow")
fig.show()
fig.write_image("./imgs/memoryUsage.png", width=1200, height=400, scale=4)